## Final dataset preparation
- Now, ones we have done cleaning and feature engineering, let's perform some final tasks :
1. Remove outliers
2. Encode categorical variables (if any)
3. standarize / normalize the data
4. Remove unnecessary columns for feeding into models

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Important thorughout this project
%matplotlib inline


In [2]:
df = pd.read_csv('../data/03_engineered/satellites_engineered.csv')

In [3]:
df.head()

,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,ORBIT_PERIOD_SEC,SEMI_MAJOR_AXIS,ORBIT_HEIGHT,PERIGEE,APOGEE,ORBITAL_SPEED,AGE_SINCE_LAUNCH,SAT_TYPE
0,13.762289,0.002461,90.2163,65.8752,347.2003,74.9038,3638,0.001091,1.074000e-05,0.0,6278.025279,7355.360594,984.360594,7337.256110,7373.465079,7.361506,61.761,C
1,13.528741,0.001678,90.2292,69.7816,244.2673,178.7129,82182,0.000110,8.200000e-07,0.0,6386.403631,7439.769906,1068.769906,7427.284484,7452.255328,7.319626,61.761,E
2,13.335738,0.006776,89.9871,212.7305,243.9611,271.6250,92607,0.000109,6.300000e-07,0.0,6478.831324,7511.379609,1140.379609,7460.481750,7562.277468,7.284652,60.761,E
3,13.362208,0.007127,89.9109,124.8412,101.8063,291.1203,92870,0.000297,1.650000e-06,0.0,6465.997033,7501.456514,1130.456514,7447.996634,7554.916394,7.289468,60.761,H
4,14.724473,0.000425,69.9186,300.3182,327.8826,32.2060,2866,0.001667,1.053100e-04,0.0,5867.782237,7031.337118,660.337118,7028.346690,7034.327546,7.529215,58.761,A


----
### 1. Removing outliers
##### Key question : Why even remove outliers if we are going to perform anomaly detection?


### 1️⃣ Garbage-in → Garbage-out

* ML models (Isolation Forest, KMeans, etc.) assume that most of your data is “normal” to learn patterns.
* If your dataset accidentally has **obvious errors** (e.g., height = 0 km, speed = 1e6 m/s, negative values), the model may learn wrong patterns or consider normal data as anomalous.

---

### 2️⃣ Helps define “normal”

* Anomaly detection models define anomalies relative to what is normal.
* If there are **obvious outliers**, your baseline “normal” distribution will be skewed, reducing detection accuracy for true anomalies like orbital maneuvers.

---

### 3️⃣ Prevents false positives

* Spotting obvious errors ensures you **don’t flag bad data as anomalies**.
* Real anomalies should reflect **interesting satellite behavior**, not just bad CSV entries or measurement glitches.

---

✅ **Summary:**

* Spotting anomalies early is about **data quality**, not about defeating your anomaly detection goal.
* Once the dataset is clean, your ML models can focus on **real, subtle anomalies** (like unusual delta in orbital height, speed, or maneuvers).




---

## ❗For prototyping purpose, we will not perform outlier removal step for now, later on we may perform that!

### 2. Encode categorical variables 

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11990 entries, 0 to 11989
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   MEAN_MOTION        11990 non-null  float64
 1   ECCENTRICITY       11990 non-null  float64
 2   INCLINATION        11990 non-null  float64
 3   RA_OF_ASC_NODE     11990 non-null  float64
 4   ARG_OF_PERICENTER  11990 non-null  float64
 5   MEAN_ANOMALY       11990 non-null  float64
 6   REV_AT_EPOCH       11990 non-null  int64  
 7   BSTAR              11990 non-null  float64
 8   MEAN_MOTION_DOT    11990 non-null  float64
 9   MEAN_MOTION_DDOT   11990 non-null  float64
 10  ORBIT_PERIOD_SEC   11990 non-null  float64
 11  SEMI_MAJOR_AXIS    11990 non-null  float64
 12  ORBIT_HEIGHT       11990 non-null  float64
 13  PERIGEE            11990 non-null  float64
 14  APOGEE             11990 non-null  float64
 15  ORBITAL_SPEED      11990 non-null  float64
 16  AGE_SINCE_LAUNCH   119

- We only have one categorical feature, that's going to be feed into the models and that is 'satellite type'.
- So we will use OneHotEncoder to encode the feature

In [5]:
from sklearn.preprocessing import OneHotEncoder

In [6]:
# Fit and transform
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_encoded = encoder.fit_transform(df[['SAT_TYPE']])

# Get actual category names
encoded_cols = encoder.get_feature_names_out(['SAT_TYPE'])

# Convert to DataFrame with proper column names
df_encoded = pd.concat([df.drop('SAT_TYPE', axis=1), pd.DataFrame(X_encoded, columns=encoded_cols)], axis=1)


In [7]:
df_encoded.head()

,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,...,SAT_TYPE_Q,SAT_TYPE_R,SAT_TYPE_S,SAT_TYPE_T,SAT_TYPE_U,SAT_TYPE_V,SAT_TYPE_W,SAT_TYPE_X,SAT_TYPE_Y,SAT_TYPE_Z
0,13.762289,0.002461,90.2163,65.8752,347.2003,74.9038,3638,0.001091,1.074000e-05,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,13.528741,0.001678,90.2292,69.7816,244.2673,178.7129,82182,0.000110,8.200000e-07,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,13.335738,0.006776,89.9871,212.7305,243.9611,271.6250,92607,0.000109,6.300000e-07,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,13.362208,0.007127,89.9109,124.8412,101.8063,291.1203,92870,0.000297,1.650000e-06,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,14.724473,0.000425,69.9186,300.3182,327.8826,32.2060,2866,0.001667,1.053100e-04,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
# We combined Main dataframe (excluding the SAT_TYPE) + (X encoded (values) + encoded_cols feature names) 
X_encoded

array([[0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.]], shape=(11990, 24))

In [9]:
encoded_cols

array(['SAT_TYPE_A', 'SAT_TYPE_B', 'SAT_TYPE_C', 'SAT_TYPE_D',
       'SAT_TYPE_E', 'SAT_TYPE_F', 'SAT_TYPE_G', 'SAT_TYPE_H',
       'SAT_TYPE_J', 'SAT_TYPE_K', 'SAT_TYPE_L', 'SAT_TYPE_M',
       'SAT_TYPE_N', 'SAT_TYPE_P', 'SAT_TYPE_Q', 'SAT_TYPE_R',
       'SAT_TYPE_S', 'SAT_TYPE_T', 'SAT_TYPE_U', 'SAT_TYPE_V',
       'SAT_TYPE_W', 'SAT_TYPE_X', 'SAT_TYPE_Y', 'SAT_TYPE_Z'],
      dtype=object)

In [10]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11990 entries, 0 to 11989
Data columns (total 41 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   MEAN_MOTION        11990 non-null  float64
 1   ECCENTRICITY       11990 non-null  float64
 2   INCLINATION        11990 non-null  float64
 3   RA_OF_ASC_NODE     11990 non-null  float64
 4   ARG_OF_PERICENTER  11990 non-null  float64
 5   MEAN_ANOMALY       11990 non-null  float64
 6   REV_AT_EPOCH       11990 non-null  int64  
 7   BSTAR              11990 non-null  float64
 8   MEAN_MOTION_DOT    11990 non-null  float64
 9   MEAN_MOTION_DDOT   11990 non-null  float64
 10  ORBIT_PERIOD_SEC   11990 non-null  float64
 11  SEMI_MAJOR_AXIS    11990 non-null  float64
 12  ORBIT_HEIGHT       11990 non-null  float64
 13  PERIGEE            11990 non-null  float64
 14  APOGEE             11990 non-null  float64
 15  ORBITAL_SPEED      11990 non-null  float64
 16  AGE_SINCE_LAUNCH   119

---
### 3. Feature Scaling 

- **Numeric Features:**  
  Features like `SEMI_MAJOR_AXIS`, `ORBIT_HEIGHT`, `ORBITAL_SPEED`, etc., have different units and ranges. Scaling them using **StandardScaler** standardizes the values (mean=0, std=1) so that all numeric features contribute equally to distance-based algorithms like **KMeans** and **Isolation Forest**.

- **Categorical Features (One-Hot Encoded):**  
  Features like `SAT_TYPE` are converted into binary columns (0/1). These **do not require scaling**, as their values are already normalized and scaling would distort the categorical meaning.

- **Key Idea:**  
  - Scale numeric continuous features.  
  - Keep one-hot categorical features as-is.  
  This ensures the model correctly interprets distances and patterns without bias from different units.


In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer # lets you apply different preprocessing to different columns

In [12]:
scaler = StandardScaler()

encoded_cols = [col for col in df_encoded.columns if col.startswith('SAT_TYPE_')]
numeric_cols = df_encoded.drop(columns = encoded_cols).columns

ct = ColumnTransformer([
    ('scaler', StandardScaler(), numeric_cols), # scale numeric feature
    ('pass', 'passthrough', encoded_cols) # keep encoded columns as is
])

df_scaled = ct.fit_transform(df_encoded)

In [13]:
df_scaled

array([[-2.00572227,  0.6522325 ,  1.30912034, ...,  0.        ,
         0.        ,  0.        ],
       [-2.38284692,  0.38749144,  1.30973296, ...,  0.        ,
         0.        ,  0.        ],
       [-2.69449962,  2.11070834,  1.29823553, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [-2.11909243,  0.07782687,  1.13246076, ...,  0.        ,
         0.        ,  0.        ],
       [-0.12314506, -0.00962016, -1.31360407, ...,  0.        ,
         0.        ,  0.        ],
       [-0.12332714,  0.03435687, -1.31320041, ...,  0.        ,
         0.        ,  0.        ]], shape=(11990, 41))

In [14]:
# convert to dataframe                        # numeric_col is a series, so convert to a list 
df_scaled = pd.DataFrame(df_scaled, columns = numeric_cols.tolist() + encoded_cols)

In [15]:
df_scaled.head()

,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,...,SAT_TYPE_Q,SAT_TYPE_R,SAT_TYPE_S,SAT_TYPE_T,SAT_TYPE_U,SAT_TYPE_V,SAT_TYPE_W,SAT_TYPE_X,SAT_TYPE_Y,SAT_TYPE_Z
0,-2.005722,0.652233,1.309120,-1.121305,1.968277,-1.274152,-0.760235,0.162405,-0.053893,-0.015606,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-2.382847,0.387491,1.309733,-1.085740,0.868905,-0.174813,5.155879,0.015232,-0.057616,-0.015606,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-2.694500,2.110708,1.298236,0.215707,0.865634,0.809127,5.941114,0.015082,-0.057688,-0.015606,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-2.651757,2.229186,1.294617,-0.584462,-0.652645,1.015582,5.960924,0.043338,-0.057305,-0.015606,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-0.452026,-0.036020,0.345174,1.013130,1.761955,-1.726322,-0.818384,0.248900,-0.018401,-0.015606,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


- Save as model ready dataset

In [16]:
df_scaled.to_csv('../data/04_scaled/satellites_scaled.csv', index=False)

- Save the encoder also (for later use in isolation forest)

In [17]:
import joblib

joblib.dump(encoder, '../data/04_scaled/encoder_sat_type.joblib')

['../data/04_scaled/encoder_sat_type.joblib']

#### Now we are ready for Machine Learning models!